# Common Libraries

In [1]:
import os, sys, shutil, mediapy
import numpy as np
from pathlib import Path
import ipywidgets as widgets
import matplotlib.pylab as plt
import xml.etree.ElementTree as ET
from IPython.display import display, clear_output

if shutil.which("nvidia-smi") is not None:
    os.environ["MUJOCO_GL"] = "egl"
import mujoco

# Custom Libraries

In [28]:
import DeepProprioception_prj_info
sys.path.append(DeepProprioception_prj_info.prj_home_path)
from config import individual_analysis_config
from Configurations.mujoco_config import get_mjc_camera_params

sys.path.append("/home/seojin/Seojin_commonTool/Module")
from vis_3D import Plotter3D
from mujoco_util import get_joints, get_body_pos, get_site_pos
from sj_datastructure import make_3d_dataset

# Params

In [3]:
model_path = "/mnt/sdb2/DeepProprioception/Myosuite/muscle_mimic/muscle_mimic.xml"

# Constants

In [4]:
model_name = Path(model_path).stem

# Load xml

In [7]:
tree = ET.parse(model_path)
root = tree.getroot()

marker_default = root.find(".//default[@class='marker']/site")
marker_group = int(marker_default.get("group"))

# Initialize model

In [26]:
# Load model
model = mujoco.MjModel.from_xml_path(model_path)
data = mujoco.MjData(model)

# Get joint information
joint_names = [mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, idx) for idx in range(model.njnt)]
joint_info_df = get_joints(model_path)
dependent_joints = joint_info_df.loc[:, joint_info_df.loc["dependent"] == True].columns.to_numpy()
independent_joints = joint_info_df.loc[:, joint_info_df.loc["independent"] == True].columns.to_numpy()

# Get body information
body_names = [mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_BODY, i) for i in range(model.nbody)]
body_pos_df = get_body_pos(model, data)
body_name_ds = make_3d_dataset(data = body_pos_df[["global_x", "global_y", "global_z"]].to_numpy()[None, :, :],
                               wrapping_dataset_name = "3D",
                               element_dataset_names = ["Time", "Label", "Coord"],
                               dataset1_dim_names = [0],
                               dataset2_dim_names = body_pos_df["body_name"].values,
                               dataset3_dim_names = ["X", "Y", "Z"])

# Render model orientation
model.vis.scale.framewidth = 0.1
model.vis.scale.framelength = 1

# Pose

In [30]:
plotter = Plotter3D()
plotter.plot_single_dataset(body_name_ds)

# Render

In [17]:
# Render
renderer = mujoco.Renderer(model, height = 400, width = 600)

# Camera
camera_params = {'azimuth': 90, 'elevation': 0, 'distance': 2.5, 'lookat': np.array([0, 0, 0])}
camera = mujoco.MjvCamera()
for key, value in camera_params.items():
    if hasattr(camera, key):
        setattr(camera, key, value)

# Scene option
scene_option = mujoco.MjvOption()
scene_option.frame = mujoco.mjtFrame.mjFRAME_WORLD
scene_option.sitegroup[:] = 0
scene_option.sitegroup[marker_group] = 1

def update(x, y, z, cam_dist, cam_azim, cam_elev):
    camera.lookat = [x, y, z]
    camera.distance = cam_dist
    camera.azimuth = cam_azim
    camera.elevation = cam_elev
    
    renderer.update_scene(mj_data, scene_option = scene_option, camera=camera)
    pixels = renderer.render()
    with output_area:
        clear_output(wait=True)
        mediapy.show_image(pixels)
        
mujoco.mj_forward(model, mj_data)

output_area = widgets.Output()
x_slider = widgets.FloatSlider(value = camera.lookat[0], min=-1.0, max=1.0, step=0.01, description='look X:')
y_slider = widgets.FloatSlider(value = camera.lookat[1], min=-1.0, max=1.0, step=0.01, description='look Y:')
z_slider = widgets.FloatSlider(value = camera.lookat[2], min=-1.0, max=1.0, step=0.01, description='look Z:')
c_dist = widgets.FloatSlider(value = camera.distance, min=0.0, max=10.0, step=0.1, description='Cam Dist', continuous_update=False)
c_azim = widgets.FloatSlider(value = camera.azimuth, min=-180, max=180, step=1, description='Cam Azim', continuous_update=False)
c_elev = widgets.FloatSlider(value = camera.elevation, min=-90, max=90, step=1, description='Cam Elev', continuous_update=False)

# Layout widgets
ui = widgets.VBox([
    widgets.HBox([x_slider, y_slider, z_slider]),
    widgets.HBox([c_dist, c_azim, c_elev]),
])

# Connect widget and function
out = widgets.interactive_output(update, {"x": x_slider, 
                                          "y": y_slider, 
                                          "z": z_slider, 
                                          "cam_dist": c_dist, 
                                          "cam_azim": c_azim, 
                                          "cam_elev": c_elev})

# Display
display(ui, output_area)

Output()